In [1]:
import os

print("OpenAI API key configured:", bool(os.getenv("OPENAI_API_KEY")))

OpenAI API key configured: True


## Document

In [2]:
document = """
Experiment 27 was conducted to study transistor behavior at different temperatures.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.

The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

The experiment concluded that transistor leakage current increases significantly at higher temperatures.
"""

## Convert it into a LangChain Document

In [3]:
from langchain_core.documents import Document

In [4]:
doc = Document(
    page_content=document,
    metadata={
        "document_id": "experiment_27",
        "document_type": "experiment"
    }
)

print(doc)

page_content='
Experiment 27 was conducted to study transistor behavior at different temperatures.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.

The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

The experiment concluded that transistor leakage current increases significantly at higher temperatures.
' metadata={'document_id': 'experiment_27', 'document_type': 'experiment'}


## LangChain Text Splitting

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Create the splitter

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

## Split our Document

In [8]:
chunks = text_splitter.split_documents([doc])

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [9]:
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)
    print("Metadata:", chunk.metadata)


--- Chunk 0 ---
Experiment 27 was conducted to study transistor behavior at different temperatures.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.
Metadata: {'document_id': 'experiment_27', 'document_type': 'experiment'}

--- Chunk 1 ---
The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.
Metadata: {'document_id': 'experiment_27', 'document_type': 'experiment'}

--- Chunk 2 ---
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.
Metadata: {'document_id': 'experiment_27', 'document_type': 'experiment'}

--- Chunk 3 ---
The experiment concluded that transistor leakage current increases significantly at higher temperatures.
Metadata: {'document_id': 'experiment_27', 'document_type': 'experiment'}


## Create the embedding model

In [10]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

## Create the Chroma vector store

In [11]:
from langchain_chroma import Chroma

In [12]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="experiment_27_langchain"
)

In [13]:
print(vectorstore._collection.count())

4


## Create the Retriever

In [14]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [15]:
question = "What happened when the temperature reached 300 degrees?"

retrieved_docs = retriever.invoke(question)

In [16]:
for rank, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Rank {rank} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Rank 1 ---
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.
Metadata: {'document_type': 'experiment', 'document_id': 'experiment_27'}

--- Rank 2 ---
The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.
Metadata: {'document_id': 'experiment_27', 'document_type': 'experiment'}

--- Rank 3 ---
Experiment 27 was conducted to study transistor behavior at different temperatures.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.
Metadata: {'document_id': 'experiment_27', 'document_type': 'experiment'}


## Create the LLM

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## Create the prompt

In [18]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
""")

## Convert retrieved documents into context

In [19]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

print(context)

The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

Experiment 27 was conducted to study transistor behavior at different temperatures.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.


## Build the prompt

In [20]:
messages = prompt.invoke({
    "context": context,
    "question": question
})

print(messages)

messages=[HumanMessage(content='\nAnswer the question using only the context below.\n\nContext:\nThe temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.\n\nThe temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.\n\nExperiment 27 was conducted to study transistor behavior at different temperatures.\n\nThe initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.\n\nQuestion:\nWhat happened when the temperature reached 300 degrees?\n\nAnswer:\n', additional_kwargs={}, response_metadata={})]


## Send it to the LLM

In [21]:
response = llm.invoke(messages)

print(response.content)

A significant increase in transistor leakage current was observed.


## Import RunnablePassthrough

In [22]:
from langchain_core.runnables import RunnablePassthrough

## Create a small helper to combine documents

In [23]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Build the RAG chain

In [24]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

## Invoke the complete chain

In [25]:
question = "What happened when the temperature reached 300 degrees?"

response = rag_chain.invoke(question)

print(response.content)

A significant increase in transistor leakage current was observed.
